In [0]:
CREATE TABLE IF NOT EXISTS workspace.myra_invest.silver_stock_news (

    symbol STRING,
    companyName STRING,
    headline STRING,
    summary STRING,
    publisher STRING,
    publishedAt TIMESTAMP,

    sentiment STRING,
    sentimentScore INT,
    confidenceScore DOUBLE,

    source STRING,
    ingestionTimestamp TIMESTAMP,
    ingestionDate DATE

)
USING DELTA;

SHOW TABLES IN workspace.myra_invest;

ALTER TABLE workspace.myra_invest.silver_stock_news
ADD COLUMNS (
    url STRING
);

In [0]:
%python

bronze_df = spark.table("workspace.myra_invest.bronze_stock_news")

print(f" Bronze Records: {bronze_df.count()}")

display(bronze_df.limit(5))

positive_keywords = [
    "profit","growth","gain","surge","record","expansion",
    "investment","partnership","strong","upgrade","bullish","acquisition"
]

negative_keywords = [
    "loss","decline","drop","fall","fraud","lawsuit",
    "penalty","investigation","downgrade","bankruptcy","weak","crash"
]

In [0]:
%python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import udf, col, concat_ws

def analyze_sentiment(text):

    text = (text or "").lower()

    positive_count = sum(1 for word in positive_keywords if word in text)
    negative_count = sum(1 for word in negative_keywords if word in text)

    if positive_count > negative_count:
        sentiment = "POSITIVE"
        score = 1
    elif negative_count > positive_count:
        sentiment = "NEGATIVE"
        score = -1
    else:
        sentiment = "NEUTRAL"
        score = 0

    confidence = abs(positive_count - negative_count) / max(
        positive_count + negative_count, 1
    )

    return (sentiment, score, round(confidence,2))

In [0]:
%python

schema = StructType([
    StructField("sentiment", StringType()),
    StructField("sentimentScore", IntegerType()),
    StructField("confidenceScore", DoubleType())
])

sentiment_udf = udf(analyze_sentiment, schema)


In [0]:
%python
silver_df = (
    bronze_df
      .withColumn(
          "combinedText",
          concat_ws(" ", col("headline"), col("summary"))
      )
      .withColumn(
          "analysis",
          sentiment_udf(col("combinedText"))
      )
      .withColumn("sentiment", col("analysis.sentiment"))
      .withColumn("sentimentScore", col("analysis.sentimentScore"))
      .withColumn("confidenceScore", col("analysis.confidenceScore"))
      .drop("analysis", "combinedText")
)

display(silver_df.limit(10))

In [0]:
%python
(silver_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("workspace.myra_invest.silver_stock_news"))

print("✅ Silver News table created successfully.")

In [0]:
%python
silver_count = spark.table("workspace.myra_invest.silver_stock_news").count()

print(f"Silver records: {silver_count}")

In [0]:
SELECT
    symbol,
    companyName,
    sentiment,
    sentimentScore,
    confidenceScore,
    headline
FROM workspace.myra_invest.silver_stock_news
LIMIT 10;